In 20260127_pairwise_to_homeo_weights.ipynb, we found the best weight for each pairwise comparison in respect to homeostatic need. We found a range of weights that can be used for each objective term without penalizing or sacrificing the homeostatic need. Now we want to find the best combination of weights for all terms together via sampling methods. We will use the same objective function as before -- homeostatic + secretion + efficiency + kinetics + diversity, but we will sample different combinations of weights for each term and evaluate the homeostatic need for each combination. We will then select the combination of weights that gives us the best homeostatic need while also maximizing the other objectives.

In [5]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [3]:
# import sim
time_num = 600
date = '2026-01-22'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [4]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

In [6]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]

def test_NetworkFlowModel(weights, solver_choice = cp.GLOP,
                          homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                          homeostatic_dm_targets = homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          maintenance=maintenance,
                          ):
    model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts, # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())), # conc
            maintenance_target=maintenance, # conc range
            kinetic_targets=np.array(list(dict(kinetic).values())), # conc range
            binary_kinetic_idx=binary_kinetics_idx,
            # force_flow_idx=force_reaction_idx,
            objective_weights=weights, #same
            upper_flux_bound= 100, # increase to 10^9 because notebook runs FlowResult using Counts if directly imported from listeners, WC runs using conc.
            target_minimal_flux=counts_to_molar[-1],
            solver=solver_choice) #SCS. ECOS, MOSEK
    return (solution.objective, solution.homeostatic_term, solution.kinetics_term, solution.efficiency_term,
            solution.secretion_term, solution.diversity_term, solution.velocities, solution.dm_dt)

In [7]:
def met_homeo(MOI, dmdt):
    ddt = dmdt.loc[MOI]
    return np.abs(ddt-homeostatic_dm_targets.loc[MOI])/homeostatic_metabolite_counts[MOI]

In [ ]:
lambda_eff_range = np.logspace(-10, -4, 1000)
lambda_kin_range = np.logspace(-10, -2, 1000)
lambda_sec_range = np.logspace(-10, -3, 1000)
lambda_div_range = np.logspace(-10, -4, 1000)

acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_eff_range:
    objective_weights = {
        "efficiency": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, kin_term, eff_term, sec_term, div_term, v, dmdt = test_NetworkFlowModel(objective_weights)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Kinetic Term'] = kin_term
        pareto[lam]['Weighted Efficiency Term'] = eff_term
        pareto[lam]['Weighted Secretion Term'] = sec_term
        pareto[lam]['Weighted Diversity Term'] = div_term
        pareto[lam]['Unweighted Kinetic Term'] = kin_term/lam
        pareto[lam]['Unweighted Efficiency Term'] = eff_term/lam
        pareto[lam]['Unweighted Secretion Term'] = sec_term/lam
        pareto[lam]['Unweighted Diversity Term'] = div_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue